# 02 - Outlier Profiling & Cleaning Rules Evaluation

This notebook tests data hygiene thresholds and edge cases on the interim venue data:
- Exact duplicate trips
- Distribution percentiles (1st, 5th, 50th, 95th, 99th, 99.9th)
- Unreasonable trip distances and durations
- Non-positive or extreme surge passenger fares

In [1]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path('../data/interim/venue_trips_2024-01.parquet')
if not DATA_PATH.exists():
    DATA_PATH = Path('data/interim/venue_trips_2024-01.parquet')

df = pd.read_parquet(DATA_PATH)
print(f'Interim baseline records loaded: {len(df):,}')

Interim baseline records loaded: 19,663,930


### 1. Duplicate Trip Analysis

In [2]:
n_duplicates = df.duplicated().sum()
print(f'Total exact duplicate rows: {n_duplicates:,}')

Total exact duplicate rows: 0


### 2. Numerical Summary Statistics & Percentile Bounds

In [3]:
metrics = ['trip_miles', 'trip_time', 'base_passenger_fare', 'driver_pay']
percentiles = [0.001, 0.01, 0.05, 0.50, 0.95, 0.99, 0.999]
df[metrics].describe(percentiles=percentiles).round(2)

,trip_miles,trip_time,base_passenger_fare,driver_pay
count,19663930.00,19663930.00,19663930.00,19663930.00
mean,4.84,1110.47,23.96,18.27
std,5.70,772.22,20.54,15.46
min,0.00,0.00,-43.09,-67.76
0.1%,0.23,113.00,5.39,4.49
1%,0.47,191.00,7.15,5.39
5%,0.75,299.00,8.17,5.41
50%,2.83,912.00,18.00,13.57
95%,16.04,2598.00,59.41,47.17
99%,26.38,3794.00,104.98,74.79


### 3. Edge Case & Anomaly Auditing
Quantifying observations that violate physical or economic sanity bounds:

In [4]:
anomalies = {
    'Zero or Negative Miles (<= 0)': (df['trip_miles'] <= 0).sum(),
    'Excessive Miles (> 100 mi)': (df['trip_miles'] > 100).sum(),
    'Sub-Minute Trips (< 60s)': (df['trip_time'] < 60).sum(),
    'Trips Over 4 Hours (> 14,400s)': (df['trip_time'] > 14400).sum(),
    'Negative or Zero Base Fare (<= $0)': (df['base_passenger_fare'] <= 0).sum(),
    'Outlier Base Fare (> $300)': (df['base_passenger_fare'] > 300).sum()
}

anomaly_df = pd.DataFrame(list(anomalies.items()), columns=['Anomaly Description', 'Record Count'])
anomaly_df['Pct of Total (%)'] = (anomaly_df['Record Count'] / len(df) * 100).round(3)
anomaly_df

,Anomaly Description,Record Count,Pct of Total (%)
0,Zero or Negative Miles (<= 0),3286,0.017
1,Excessive Miles (> 100 mi),1660,0.008
2,Sub-Minute Trips (< 60s),1640,0.008
3,"Trips Over 4 Hours (> 14,400s)",303,0.002
4,Negative or Zero Base Fare (<= $0),7334,0.037
5,Outlier Base Fare (> $300),3006,0.015


### 4. Cleaning Rule Dry Run
Testing the filtering rules intended for `src/clean.py`.

In [5]:
# Apply proposed pipeline filters
clean_mask = (
    (df['trip_miles'] > 0) &
    (df['trip_miles'] <= 100) &
    (df['trip_time'] >= 60) &
    (df['trip_time'] <= 14400) &
    (df['base_passenger_fare'] > 0)
)

cleaned_df = df[clean_mask].drop_duplicates()

removed_rows = len(df) - len(cleaned_df)
print(f'Original Records: {len(df):,}')
print(f'Cleaned Records:  {len(cleaned_df):,}')
print(f'Rows Dropped:     {removed_rows:,} ({removed_rows / len(df) * 100:.2f}%)')

Original Records: 19,663,930
Cleaned Records:  19,650,285
Rows Dropped:     13,645 (0.07%)
